# Example: Production Portfolio Engine

In this example, we run the full production loop on real Alpaca data: bandit asset selection, EWLS parameter updates, sentiment-driven lambda override, escalation triggers, and live order execution. We build on the paper trading positions from Session 3 and add the production decision layers that transform a validated strategy into an operating system.

> __Learning Objectives:__
>
> * __Run the live production loop:__ Execute the bandit asset selector and Cobb-Douglas allocator on real Alpaca positions with EWLS-updated SIM parameters. Observe how the system generates target allocations from live market state and submits orders.
> * __Implement sentiment monitoring with override:__ Compute real-time sentiment from recent price momentum and configure automatic lambda override when markets deteriorate. Evaluate the portfolio weight shift when sentiment drops below the defensive threshold.
> * __Define and test escalation triggers:__ Set up drawdown, sentiment crash, and bandit churn escalation rules on real account state from Alpaca. Inspect the audit trail to verify the system responds correctly to each condition.

## Setup, Data and Prerequisites

In [ ]:
include("Include.jl");

### Constants


In [ ]:
# Production-engine configuration
CREDENTIALS_PATH = joinpath(_ROOT, "config", "credentials.toml")
ALPACA_SECTION   = "Credentials"  # named section in credentials.toml; change to route to a different paper account
PRODUCTION_DATA_PATH = joinpath(_PATH_TO_DATA, "production-engine-results.jld2")
PRODUCTION_CONFIG_PATH = joinpath(_ROOT, "config", "production-config.toml")
Δt = 1.0 / 252.0
rf = 0.05
B₀ = 10_000.0
N_short = 21
N_long = 63
GAIN = 10.0
N_growth = 10
offset = 63
default_half_life = 63.0
ALLOCATION_EPSILON = 0.1
MAX_DRAWDOWN = 0.15
MAX_TURNOVER = 0.50
SENTIMENT_THRESHOLD = -0.5
SENTIMENT_OVERRIDE_LAMBDA = 2.0
MAX_BANDIT_CHURN = 2
HISTORY_LOOKBACK_YEARS = 1
ORDER_POLL_ATTEMPTS = 30
ORDER_POLL_INTERVAL_SECONDS = 1.0


We connect to Alpaca, check the market clock, and determine whether we are in live or review mode.

In [ ]:
client, is_live, production_data_path = let
    # --- Step 1: Load Alpaca credentials ---
    creds_path = CREDENTIALS_PATH;
    if !isfile(creds_path)
        error("Credentials file not found at $(creds_path).\n" *
              "Copy config/credentials.toml.example to config/credentials.toml " *
              "and fill in your Alpaca paper trading API keys.")
    end
    client = Alpaca.load_client(creds_path; section = ALPACA_SECTION);

    # --- Verify which Alpaca account we are actually connected to ---
    acct = Alpaca.get_account(client);
    println("Connected account: $(acct.account_number)  status=$(acct.status)  ($(ALPACA_SECTION))");

    # --- Step 2: Check market clock ---
    clock = Alpaca.get_clock(client);
    println("Market status: $(clock.is_open ? "OPEN" : "CLOSED")")
    println("  Server time : $(clock.timestamp)")
    println("  Next open   : $(clock.next_open)")
    println("  Next close  : $(clock.next_close)")

    # --- Step 3: Determine execution mode ---
    is_live = clock.is_open;
    production_data_path = PRODUCTION_DATA_PATH;
    client, is_live, production_data_path
end


Next we define model constants, resolve the ticker universe from Session 1, and load pre-calibrated SIM parameters.

In [ ]:
my_tickers, K, sim_params, prod_ctx = let
    # --- Step 2: Resolve ticker universe ---
    # Priority: Session 1 min-variance portfolio → production-config.toml fallback
    s1_path = joinpath(_PATH_TO_DATA_S1, "minvar-allocation.jld2");
    config_path = PRODUCTION_CONFIG_PATH;
    cfg = TOML.parsefile(config_path);
    ticker_source = get(cfg["Tickers"], "source", "session-1");

    if ticker_source == "session-1" && isfile(s1_path)
        s1_data = load(s1_path);
        my_tickers = s1_data["my_tickers"];
        println("Loaded $(length(my_tickers)) tickers from Session 1 portfolio: $(my_tickers)");
    elseif ticker_source == "session-1" && !isfile(s1_path)
        my_tickers = String.(cfg["Tickers"]["universe"]);
        @warn "Session 1 artifact not found at $(s1_path). Falling back to production-config.toml: $(my_tickers)"
    else
        my_tickers = String.(cfg["Tickers"]["universe"]);
        println("Manual ticker universe from production-config.toml: $(my_tickers)");
    end
    K = length(my_tickers);

    # --- Step 3: Load pre-calibrated SIM parameters ---
    calibration = load(joinpath(eCornellAIFinance._PATH_TO_DATA, "sim-calibration.jld2"));
    all_tickers = calibration["tickers"];
    all_alpha = calibration["alpha"];
    all_beta = calibration["beta"];
    all_sigma = calibration["sigma_eps"];
    ticker_idx = Dict(all_tickers[i] => i for i in eachindex(all_tickers));
    for t in my_tickers
        haskey(ticker_idx, t) || error("Ticker $(t) not found in calibration set.");
    end
    sim_params = Dict{String,Tuple{Float64,Float64,Float64}}(
        t => (all_alpha[ticker_idx[t]], all_beta[ticker_idx[t]], all_sigma[ticker_idx[t]])
        for t in my_tickers
    );

    # --- Step 4: Auto-write resolved tickers to production-config.toml ---
    cfg["Tickers"]["universe"] = my_tickers;
    open(config_path, "w") do io
        TOML.print(io, cfg);
    end
    println("production-config.toml updated with $(K)-ticker universe.");

    # --- Step 5: Build production context ---
    prod_ctx = build(MyProductionContext, (
        tickers = my_tickers,
        sim_parameters = sim_params,
        B₀ = B₀,
        epsilon = ALLOCATION_EPSILON,
        max_drawdown = MAX_DRAWDOWN,
        max_turnover = MAX_TURNOVER,
        sentiment_threshold = SENTIMENT_THRESHOLD,
        sentiment_override_lambda = SENTIMENT_OVERRIDE_LAMBDA,
        max_bandit_churn = MAX_BANDIT_CHURN
    ));

    println("Production context: $(K) tickers, max DD = $(prod_ctx.max_drawdown*100)%, sentiment threshold = $(prod_ctx.sentiment_threshold)");
    my_tickers, K, sim_params, prod_ctx
end


We fetch one year of historical bars and the current account positions from Alpaca.

In [ ]:
market_prices, price_matrix, current_shares, current_cash, current_equity = let
    # --- Step 1: Fetch historical bars ---
    finish_date = Dates.format(today(), "yyyy-mm-dd");
    start_date = Dates.format(today() - Year(HISTORY_LOOKBACK_YEARS), "yyyy-mm-dd");
    symbols = vcat(my_tickers, ["SPY"]);
    println("Fetching daily bars for $(length(symbols)) symbols: $(start_date) to $(finish_date)...");
    bars_dict = Alpaca.get_bars(client, symbols, "1Day"; start = start_date, finish = finish_date);

    spy_bars = bars_dict["SPY"];
    n_days = length(spy_bars);
    market_prices = [b.c for b in spy_bars];

    price_matrix = zeros(n_days, K + 1);
    price_matrix[:, 1] = 1:n_days;
    for (k, ticker) in enumerate(my_tickers)
        ticker_bars = bars_dict[ticker];
        n_ticker = min(length(ticker_bars), n_days);
        for i in 1:n_ticker
            price_matrix[i, k + 1] = ticker_bars[i].c;
        end
    end

    # --- Step 2: Get current positions from Alpaca ---
    positions = Alpaca.list_positions(client);
    current_shares = zeros(K);
    for (k, ticker) in enumerate(my_tickers)
        for pos in positions
            if pos.symbol == ticker
                current_shares[k] = pos.qty;
                break;
            end
        end
    end

    acct = Alpaca.get_account(client);
    current_cash = acct.cash;
    current_equity = acct.equity;

    println("Fetched $(n_days) trading days. Current equity: \$$(round(current_equity, digits=2))");
    market_prices, price_matrix, current_shares, current_cash, current_equity
end


---
## Task 1: Run the Live Production Loop

> __The production step:__
>
> Each trading day, the system updates its SIM parameter estimates via EWLS, computes sentiment from recent price momentum, runs the bandit to select which assets to include, checks all escalation triggers, and — if safe — allocates via Cobb-Douglas and submits orders. This cell executes one such step on today's data.

We initialize EWLS from pre-calibrated parameters, warm up on historical bars, and execute the production step.

In [ ]:
ewls_states, prev_action, peak_wealth, production_history, event_history, today_result, today_events = let
    # --- Step 1: Initialize EWLS from pre-calibrated params ---
    ewls_states = Dict{String,MyEWLSState}(
        t => ewls_init(sim_params[t]...; half_life = default_half_life) for t in my_tickers
    );

    # --- Step 2: Warm up EWLS on historical bars ---
    gm_raw = compute_market_growth(market_prices; Δt = Δt);
    for t_idx in 1:min(offset - 1, length(gm_raw))
        gm_t = gm_raw[t_idx];
        for (k, ticker) in enumerate(my_tickers)
            if t_idx + 1 <= size(price_matrix, 1)
                gi_t = (1.0 / Δt) * log(price_matrix[t_idx + 1, k + 1] / price_matrix[t_idx, k + 1]);
                ewls_update!(ewls_states[ticker], gi_t, gm_t);
            end
        end
    end

    # continue EWLS through remaining historical data
    for t_idx in offset:length(gm_raw)
        gm_t = gm_raw[t_idx];
        for (k, ticker) in enumerate(my_tickers)
            if t_idx + 1 <= size(price_matrix, 1)
                gi_t = (1.0 / Δt) * log(price_matrix[t_idx + 1, k + 1] / price_matrix[t_idx, k + 1]);
                ewls_update!(ewls_states[ticker], gi_t, gm_t);
            end
        end
    end

    # --- Step 3: Load previous production history (if any) ---
    prev_action = ones(Int, K);
    peak_wealth = max(B₀, current_equity);
    production_history = MyLiveProductionDayResult[];
    event_history = MyEscalationEvent[];
    if isfile(production_data_path)
        saved = load(production_data_path);
        if haskey(saved, "history")
            production_history = saved["history"];
            if !isempty(production_history)
                prev_action = production_history[end].bandit_action;
                peak_wealth = max(peak_wealth, maximum(r.wealth for r in production_history));
            end
        end
        if haskey(saved, "events")
            event_history = saved["events"];
        end
    end

    # --- Step 4: Run production step ---
    n_days = size(price_matrix, 1);
    println("Running production step (day $(n_days))...");
    (today_result, today_events) = run_production_step(
        prod_ctx, ewls_states, price_matrix, market_prices, my_tickers, n_days;
        n_bandit_iters = 100, prev_action = prev_action,
        peak_wealth = peak_wealth, current_shares = current_shares,
        current_cash = current_cash, N_short = N_short, N_long = N_long,
        GAIN = GAIN, N_growth = N_growth
    );
    today_result.timestamp = string(now());

    println("Production step complete:");
    println("  Sentiment: $(round(today_result.sentiment, digits=3))");
    println("  Lambda (effective): $(round(today_result.lambda, digits=3))");
    println("  Bandit selected: $(sum(today_result.bandit_action))/$(K) assets");
    println("  Escalated: $(today_result.escalated)");
    println("  Rebalanced: $(today_result.rebalanced)");
    ewls_states, prev_action, peak_wealth, production_history, event_history, today_result, today_events
end


If the market is open, we submit orders to Alpaca. Otherwise we load saved results from the last live run.

In [ ]:
order_ids, today_result, today_events = let
    if is_live && today_result.rebalanced
        # === LIVE MODE: Submit Orders ===
        order_ids = String[];
        for (k, ticker) in enumerate(my_tickers)
            delta = today_result.shares[k] - current_shares[k];
            if abs(delta) < 0.01
                continue;
            end
            side = delta > 0 ? "buy" : "sell";
            qty = round(abs(delta), digits=2);
            println("Submitting $(uppercase(side)) $(qty) $(ticker)...");
            order = Alpaca.submit_order(client, ticker, qty, side;
                type = "market", time_in_force = "day");
            push!(order_ids, order.id);

            # poll for fill
            for attempt in 1:ORDER_POLL_ATTEMPTS
                sleep(ORDER_POLL_INTERVAL_SECONDS);
                updated = Alpaca.get_order(client, order.id);
                if updated.status == "filled"
                    println("  $(ticker): FILLED at \$$(round(updated.filled_avg_price, digits=2))");
                    break;
                end
            end
        end
        today_result.order_ids = order_ids;
    elseif !is_live && isfile(production_data_path)
        saved = load(production_data_path);
        if haskey(saved, "latest_result")
            today_result = saved["latest_result"];
            today_events = haskey(saved, "latest_events") ? saved["latest_events"] : MyEscalationEvent[];
            println("REVIEW MODE: Loaded production step from $(today_result.timestamp)");
        end
    end
    order_ids, today_result, today_events
end


We display the production decision table showing EWLS parameters, bandit selection, and target vs current shares.

In [ ]:
let
    # --- Production Decision Summary ---
    df = DataFrame(
        "Ticker" => my_tickers,
        "EWLS α" => [round(today_result.ewls_params[t][1], digits=6) for t in my_tickers],
        "EWLS β" => [round(today_result.ewls_params[t][2], digits=4) for t in my_tickers],
        "Bandit" => [today_result.bandit_action[k] == 1 ? "IN" : "OUT" for k in 1:K],
        "Target" => [round(today_result.shares[k], digits=2) for k in 1:K],
        "Current" => [round(current_shares[k], digits=2) for k in 1:K],
        "Δ" => [round(today_result.shares[k] - current_shares[k], digits=2) for k in 1:K]
    );
    println("Production Decision ($(today_result.timestamp)):");
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))

    println("\nSentiment: $(round(today_result.sentiment, digits=3)) | Lambda: $(round(today_result.lambda, digits=3)) | Wealth: \$$(round(today_result.wealth, digits=2))");
end

We replay the engine over the full historical window to visualize wealth and EWLS parameter drift.

In [ ]:
let
    # --- Step 1: Run replay to get historical wealth estimate ---
    replay = replay_engine_ewls(price_matrix, market_prices, my_tickers, sim_params,
        (max_drawdown = prod_ctx.max_drawdown, max_turnover = prod_ctx.max_turnover);
        B₀ = B₀, offset = offset, half_life = default_half_life,
        N_short = N_short, N_long = N_long, GAIN = GAIN, N_growth = N_growth,
        cost_bps = 5.0, epsilon = 0.1);

    n_trading = length(replay.wealth);
    days = 0:(n_trading - 1);

    # --- Step 2: Panel 1 — Wealth trajectory (W/W₀) ---
    p1 = plot(days, replay.wealth ./ B₀, label="CD Engine (EWLS)", linewidth=2, color=:steelblue,
        xlabel="Trading Day", ylabel="W/W₀", title="Historical Engine Performance",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    hline!(p1, [1.0], label="", linestyle=:dash, color=:gray, alpha=0.3);

    # --- Step 3: Panel 2 — β drift ---
    p2 = plot(title="SIM β Drift (EWLS)", xlabel="Trading Day", ylabel="β",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    for k in 1:min(3, K)
        ticker = my_tickers[k];
        β_hist = [p[2] for p in replay.param_history[ticker]];
        plot!(p2, 0:(length(β_hist)-1), β_hist, label=ticker, linewidth=1.5);
    end

    plot(p1, p2, layout=(1, 2), size=(900, 400))
end

---
## Task 2: Sentiment Monitoring with Lambda Override

> __Proactive defense:__
>
> The sentiment signal detects deteriorating market conditions from price momentum before a drawdown fully materializes. When sentiment drops below the threshold, the system overrides the EMA-based lambda with a high defensive value, shifting allocation toward lower-beta assets.

We compute the sentiment signal over the full historical window and check whether the override is active today.

In [ ]:
sentiment_series = let
    # --- Compute sentiment over full history ---
    n_days = length(market_prices);
    sentiment_series = zeros(n_days);
    for t in 6:n_days
        ret_5d = market_prices[t] / market_prices[t - 5] - 1.0;
        sentiment_series[t] = tanh(10.0 * ret_5d);
    end

    s_current = today_result.sentiment;
    threshold = prod_ctx.sentiment_threshold;
    override_active = s_current < threshold;

    println("Current sentiment: $(round(s_current, digits=3))");
    println("Threshold: $(threshold)");
    println("Override active: $(override_active ? "YES → λ = $(prod_ctx.sentiment_override_lambda)" : "NO → λ from EMA")");
    println("Effective λ: $(round(today_result.lambda, digits=3))");
    sentiment_series
end


We plot the sentiment signal, effective lambda, and market price over the historical window.

In [ ]:
let
    n = length(sentiment_series);

    # --- Step 1: Compute lambda series ---
    ema_s = compute_ema(market_prices; window = N_short);
    ema_l = compute_ema(market_prices; window = N_long);
    λ_series = compute_lambda(ema_s, ema_l; G = GAIN);

    # --- Step 2: Effective lambda with override ---
    λ_eff = copy(λ_series);
    for t in 1:n
        if sentiment_series[t] < prod_ctx.sentiment_threshold
            λ_eff[t] = prod_ctx.sentiment_override_lambda;
        end
    end

    # --- Step 3: Panel 1 — Sentiment signal ---
    p1 = plot(1:n, sentiment_series, label="Sentiment", linewidth=1.5, color=:steelblue,
        xlabel="Day", ylabel="s_t", title="Sentiment Signal",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    hline!(p1, [prod_ctx.sentiment_threshold], label="Threshold", linestyle=:dash, color=:red, linewidth=1.5);

    # --- Step 4: Panel 2 — Lambda with override ---
    p2 = plot(1:n, λ_eff, label="Effective λ", linewidth=1.5, color=:darkorange,
        xlabel="Day", ylabel="λ", title="Lambda (with override)",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);
    plot!(p2, 1:n, λ_series, label="Base λ (EMA)", linewidth=1, color=:gray, alpha=0.5);

    # --- Step 5: Panel 3 — Market price ---
    p3 = plot(1:n, market_prices ./ market_prices[1], label="SPY (P/P₀)", linewidth=1.5, color=:black,
        xlabel="Day", ylabel="P/P₀", title="Market",
        bg="gray95", background_color_outside="white", framestyle=:box, fg_legend=:transparent);

    plot(p1, p2, p3, layout=(3, 1), size=(800, 700))
end

We sweep the sentiment threshold to see how it controls the tradeoff between false alarms and missed signals.

In [ ]:
let
    thresholds = [-0.3, -0.4, -0.5, -0.6, -0.7];
    n = length(sentiment_series);
    results_sweep = [];
    for thresh in thresholds
        override_days = sum(sentiment_series[offset+1:end] .< thresh);
        trading_days = n - offset;
        override_pct = round(override_days / trading_days * 100, digits=1);
        push!(results_sweep, (thresh, override_days, override_pct));
    end

    df = DataFrame(
        "Threshold" => [r[1] for r in results_sweep],
        "Override Days" => [r[2] for r in results_sweep],
        "Override (%)" => [r[3] for r in results_sweep]
    );
    println("Sentiment Threshold Sensitivity:");
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

---
## Task 3: Escalation Triggers and Audit Trail

> __Three-layer safety net:__
>
> The escalation system checks drawdown (critical — de-risk to cash), sentiment crash (warning — override lambda), and bandit churn (warning — hold previous allocation) on every production step. Each trigger fires independently, and all events are logged with timestamps for investment committee review.

We compute trigger conditions from the current production state and display their status.

In [ ]:
let
    # --- Compute trigger conditions ---
    drawdown = peak_wealth > 0 ? (peak_wealth - today_result.wealth) / peak_wealth : 0.0;
    churn = sum(abs.(today_result.bandit_action .- prev_action));

    df = DataFrame(
        "Trigger" => ["Drawdown", "Sentiment Crash", "Bandit Churn"],
        "Condition" => [
            "(peak - W) / peak",
            "s_t < threshold",
            "Σ|a_t - a_{t-1}|"
        ],
        "Threshold" => [
            "$(round(prod_ctx.max_drawdown * 100, digits=0))%",
            "$(prod_ctx.sentiment_threshold)",
            "$(prod_ctx.max_bandit_churn)"
        ],
        "Actual" => [
            "$(round(drawdown * 100, digits=1))%",
            "$(round(today_result.sentiment, digits=3))",
            "$(churn)"
        ],
        "Status" => [
            drawdown > prod_ctx.max_drawdown ? "CRITICAL" : "OK",
            today_result.sentiment < prod_ctx.sentiment_threshold ? "WARNING" : "OK",
            churn > prod_ctx.max_bandit_churn ? "WARNING" : "OK"
        ]
    );
    println("Escalation Trigger Status:");
    pretty_table(df; backend = :text,
        fit_table_in_display_horizontally = false,
        fit_table_in_display_vertically = false,
        table_format = TextTableFormat(borders = text_table_borders__compact))
end

We display the cumulative escalation log from all production runs.

In [ ]:
let
    all_events = vcat(event_history, today_events);
    if isempty(all_events)
        println("No escalation events recorded.")
    else
        df = DataFrame(
            "Day" => [e.day for e in all_events],
            "Trigger" => [e.trigger_type for e in all_events],
            "Severity" => [string(e.severity) for e in all_events],
            "Message" => [e.message for e in all_events],
            "Action" => [e.recommended_action for e in all_events]
        );
        println("Escalation Log ($(length(all_events)) events):");
        pretty_table(df; backend = :text,
            fit_table_in_display_horizontally = false,
            fit_table_in_display_vertically = false,
            table_format = TextTableFormat(borders = text_table_borders__compact))
    end
end

We save the production results to disk so the next run can pick up the history.

In [ ]:
let
    if is_live
        # append today's result to history
        push!(production_history, today_result);
        append!(event_history, today_events);

        save(production_data_path, Dict(
            "history" => production_history,
            "events" => event_history,
            "latest_result" => today_result,
            "latest_events" => today_events,
            "tickers" => my_tickers,
            "sim_params" => sim_params,
            "timestamp" => string(now())
        ));
        println("Production results saved to $(production_data_path)");
        println("History: $(length(production_history)) production days recorded.");
    else
        println("Review mode — no data saved.");
    end
end

---
## Summary

This example executed the full production portfolio engine on real Alpaca data, from EWLS parameter updates through order submission.

### Key Takeaways
* __The production step combines four decision layers:__ EWLS parameter updates, bandit asset selection, sentiment-driven lambda override, and escalation trigger checking all execute before a single order is placed. Each layer operates on real Alpaca data — live prices, live account equity, live positions.
* __Sentiment override is the first line of defense:__ When the 5-day return momentum drops below the threshold, the system shifts to a defensive posture before the drawdown trigger fires. The sensitivity sweep shows how the threshold controls the tradeoff between false alarms and missed signals.
* __The escalation log is an audit trail:__ Every trigger event is recorded with its day, severity, message, and recommended action. This log is the artifact that an investment committee would review to verify the system is operating within its risk mandate.

### Disclaimer
This content is for educational purposes only and does not constitute investment advice.